In [ ]:
import numpy as np
import pandas as pd
import copy
import os, sys
import glob
import re
import datetime as dt
import time
from timethis import timethis
import subprocess
import importlib
from common_params import parent_directory_images, parent_directory_url_csvs
from preprocess_data_fns import (
    create_image_files_df, get_predicted_categories_clip, categorize_and_plot, categories,
    crop_out_text, banner_text_box, preprocess_image_for_banner_text, body_style_dict

)


In [1]:
import os,sys
os.chdir('/Users/levgolod/Projects/car_classifier/')
from _01_collect_data.find_vehicle_image_urls import *

In [2]:
df = compile_image_urls_df()
df.head()

,vehicle_image_url,vehicle_id,url,vin,year_make_model,list_price,listing_details,listing_narrative,filename
0,https://images.autotrader.com/scaler/500/375/h...,696976212,https://www.autotrader.com/cars-for-sale/vehic...,JHMCP26358C070723,Used 2008 Honda Accord LX,"$6,900","238,838 miles^2.4L 4-Cylinder Gas Engine^21 Ci...",* All records in possession^* Regularly mainta...,696976212.csv
1,https://images.autotrader.com/scaler/500/375/h...,696976212,https://www.autotrader.com/cars-for-sale/vehic...,JHMCP26358C070723,Used 2008 Honda Accord LX,"$6,900","238,838 miles^2.4L 4-Cylinder Gas Engine^21 Ci...",* All records in possession^* Regularly mainta...,696976212.csv
2,https://images.autotrader.com/scaler/500/375/h...,696976212,https://www.autotrader.com/cars-for-sale/vehic...,JHMCP26358C070723,Used 2008 Honda Accord LX,"$6,900","238,838 miles^2.4L 4-Cylinder Gas Engine^21 Ci...",* All records in possession^* Regularly mainta...,696976212.csv
3,https://images.autotrader.com/scaler/500/375/h...,696976212,https://www.autotrader.com/cars-for-sale/vehic...,JHMCP26358C070723,Used 2008 Honda Accord LX,"$6,900","238,838 miles^2.4L 4-Cylinder Gas Engine^21 Ci...",* All records in possession^* Regularly mainta...,696976212.csv
4,https://images.autotrader.com/scaler/500/375/h...,696976212,https://www.autotrader.com/cars-for-sale/vehic...,JHMCP26358C070723,Used 2008 Honda Accord LX,"$6,900","238,838 miles^2.4L 4-Cylinder Gas Engine^21 Ci...",* All records in possession^* Regularly mainta...,696976212.csv


In [15]:
print(df.shape)
df = df.drop_duplicates(subset=['vehicle_id'])
print(df.shape)


(27339, 10)
(1610, 10)


In [16]:
# check out unique chars
pd.Series(list("".join(df['list_price'].dropna()))).value_counts()

9    678
$    650
,    650
0    428
5    345
2    325
8    288
1    274
3    270
4    264
7    179
6    159
i      6
       2
e      2
c      2
r      2
P      2
L      2
g      2
n      2
t      2
s      2
^      2
Name: count, dtype: int64

In [17]:
def try_float(x:str):
    try:
        return float(str(x))
    except:
        return None

def clean_price(price_str: str):
    cleaned = re.sub(r"[^\d.]", "", str(price_str))
    if all([
        cleaned.isnumeric(),
        # cleaned   != ''
    ]):
        return try_float(cleaned)
    else:
        return None


In [18]:
df['price_float'] = df['list_price'].apply(clean_price)

In [19]:
# df['list_price'].head().apply(clean_price)
#
# df['price_float'] = None
# for i in df.index:
#     try:
#         df.loc[i, 'price_float'] = clean_price(df.loc[i, 'list_price'])
#     except:
#         continue
#         # print(df.loc[i, 'list_price'])



In [22]:
# df[['list_price','price_float']]
print(df['price_float'].isnull().mean())
print(df['list_price'].isnull().mean())
df.loc[df['price_float'].isnull(),  ['list_price','price_float']].drop_duplicates()

0.5962732919254659
0.5962732919254659


,list_price,price_float
24,NaN,NaN


In [23]:
df = df.loc[df['list_price'].notnull(), ]
df.shape

(650, 10)

In [25]:
df.loc[0]

vehicle_image_url    https://images.autotrader.com/scaler/500/375/h...
vehicle_id                                                   696976212
url                  https://www.autotrader.com/cars-for-sale/vehic...
vin                                                  JHMCP26358C070723
year_make_model                              Used 2008 Honda Accord LX
list_price                                                      $6,900
listing_details      238,838 miles^2.4L 4-Cylinder Gas Engine^21 Ci...
listing_narrative    * All records in possession^* Regularly mainta...
filename                                                 696976212.csv
price_float                                                     6900.0
Name: 0, dtype: object